# MPC Controller Validation Tests

This notebook validates the MPC controller implementation against the requirements
specified in `docs/design/mpc-controller-design-v1-revised.md`.

## Test Suite

1. **Aggressive-in-the-Money Behavior** - Verify MPC trades more when price is favorable
2. **Trending Path Robustness** - Ensure delta doesn't blow out when κ is misspecified
3. **Cost Sensitivity** - Higher costs → fewer trades
4. **Constraint Stress** - Controller respects position limits


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
import sys
sys.path.insert(0, '..')

# Setup logging FIRST
from src.utils.logging import setup_logging, LogConfig, TestLogger, get_logger

# Configure logging to write to logs folder
log_config = LogConfig(
    log_dir=Path("../logs"),
    console_level="INFO",
    file_level="DEBUG",
)
setup_logging(log_config)

# Now import the rest
from src.control import DeltaHedgeMPC, MPCConfig, MPCState, SimpleKappaEstimator

# Set random seed for reproducibility
np.random.seed(42)

# Create test logger for this validation run
test_logger = TestLogger("MPC_Validation_v1")
test_logger.log_test_start("Validating MPC controller against design spec")


## Helper Functions


In [ ]:
def generate_ou_path(S0: float, S_bar: float, kappa: float, sigma: float, 
                     dt: float, n_steps: int) -> np.ndarray:
    """
    Generate Ornstein-Uhlenbeck price path.
    
    dS = κ(S̄ - S)dt + σS√dt dW
    """
    S = np.zeros(n_steps)
    S[0] = S0
    
    for t in range(1, n_steps):
        dW = np.random.randn() * np.sqrt(dt)
        S[t] = S[t-1] + kappa * (S_bar - S[t-1]) * dt + sigma * S[t-1] * dW
    
    return S


def generate_trending_path(S0: float, drift: float, sigma: float,
                           dt: float, n_steps: int) -> np.ndarray:
    """
    Generate trending (GBM) price path with no mean reversion.
    
    dS = μS dt + σS dW
    """
    S = np.zeros(n_steps)
    S[0] = S0
    
    for t in range(1, n_steps):
        dW = np.random.randn() * np.sqrt(dt)
        S[t] = S[t-1] * (1 + drift * dt + sigma * dW)
    
    return S


def compute_ema(prices: np.ndarray, span: int) -> np.ndarray:
    """Compute exponential moving average."""
    alpha = 2.0 / (span + 1)
    ema = np.zeros_like(prices)
    ema[0] = prices[0]
    for t in range(1, len(prices)):
        ema[t] = alpha * prices[t] + (1 - alpha) * ema[t-1]
    return ema


def simulate_gamma_scalping(prices: np.ndarray, mpc: DeltaHedgeMPC,
                           gamma_opt: float = 37.0, sigma: float = 0.15,
                           initial_delta_opt: float = 0.0,
                           initial_q: float = 0.0) -> dict:
    """
    Simulate MPC-based gamma scalping on a price path.
    
    Returns dict with trajectories and statistics.
    """
    n_steps = len(prices)
    ema = compute_ema(prices, span=20)
    
    # State tracking
    delta_opt = np.zeros(n_steps)
    q = np.zeros(n_steps)
    delta_portfolio = np.zeros(n_steps)
    trades = np.zeros(n_steps)
    
    # Initial conditions
    delta_opt[0] = initial_delta_opt
    q[0] = initial_q
    delta_portfolio[0] = delta_opt[0] + q[0]
    
    for t in range(1, n_steps):
        # Options delta evolves with gamma
        dS = prices[t] - prices[t-1]
        delta_opt[t] = delta_opt[t-1] + gamma_opt * dS
        
        # Current state for MPC
        state = MPCState(
            S=prices[t],
            delta_opt=delta_opt[t],
            q=q[t-1],  # Position before trade
            gamma_opt=gamma_opt,
            sigma=sigma,
            S_bar=ema[t]
        )
        
        # Solve MPC
        solution = mpc.solve(state)
        
        # Execute trade
        trades[t] = solution.u_optimal
        q[t] = q[t-1] + trades[t]
        delta_portfolio[t] = delta_opt[t] + q[t]
    
    return {
        'prices': prices,
        'ema': ema,
        'delta_opt': delta_opt,
        'q': q,
        'delta_portfolio': delta_portfolio,
        'trades': trades,
        'n_trades': np.sum(np.abs(trades) > 0.1),
        'total_traded': np.sum(np.abs(trades)),
        'max_abs_delta': np.max(np.abs(delta_portfolio)),
        'max_q': np.max(np.abs(q)),
    }

print("Helper functions defined ✓")


## Test 1: Aggressive-in-the-Money Behavior

**Hypothesis:** When price deviates from S̄, MPC should trade more aggressively to take advantage of expected reversion.

- When S < S̄ and we need to buy (Δ < 0): trade more aggressively
- When S > S̄ and we need to sell (Δ > 0): trade more aggressively

**Pass Criteria:** Correlation between |S - S̄| × sign(trade needed) and |u| should be positive


In [ ]:
print("\n" + "="*50)
print("TEST 1: Aggressive-in-the-Money Behavior")
print("="*50)

# Parameters
S0 = 500
S_bar = 500
kappa_true = 0.3
sigma = 0.15
dt = 1.0 / (252 * 6.5 * 60)  # 1 minute
n_steps = 500

# Generate OU path
prices = generate_ou_path(S0, S_bar, kappa_true, sigma, dt, n_steps)

# Create MPC with matching kappa
config = MPCConfig(kappa=kappa_true)
mpc = DeltaHedgeMPC(config)

# Simulate
results = simulate_gamma_scalping(prices, mpc, gamma_opt=37.0)

# Analyze: Is MPC more aggressive when price is favorable?
deviation = prices - results['ema']
delta_sign = np.sign(results['delta_portfolio'][:-1])  # Direction we need to hedge

# Favorable = when price deviates in the direction we want to trade
favorable_indicator = deviation[:-1] * delta_sign  # Positive when favorable

# Should see larger trades when favorable
trade_sizes = np.abs(results['trades'][1:])

# Only look at times when there was actual delta exposure
mask = np.abs(results['delta_portfolio'][:-1]) > 10
if np.sum(mask) > 10:
    correlation = np.corrcoef(favorable_indicator[mask], trade_sizes[mask])[0, 1]
else:
    correlation = 0.0

# Log metrics
test_logger.log_metric("Total trades", results['n_trades'])
test_logger.log_metric("Total volume", results['total_traded'], "shares")
test_logger.log_metric("Max |Δ_portfolio|", results['max_abs_delta'], "shares")
test_logger.log_metric("Favorability-Size Correlation", correlation)

# Log test result
test1_pass = correlation > 0
test_logger.log_case(
    "Aggressive-in-the-Money",
    passed=test1_pass,
    expected="correlation > 0",
    actual=f"correlation = {correlation:.3f}",
    details="MPC should trade more when price is favorable for mean-reversion"
)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(prices, label='Price', color='steelblue')
axes[0].plot(results['ema'], label='EMA (S̄)', color='orange', linestyle='--')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].set_title('Test 1: OU Path with Mean-Reversion')

axes[1].plot(results['delta_portfolio'], label='Δ_portfolio', color='red')
axes[1].plot(results['delta_opt'], label='Δ_opt', color='gray', alpha=0.5)
axes[1].axhline(y=0, color='black', linestyle=':')
axes[1].set_ylabel('Delta (shares)')
axes[1].legend()

axes[2].bar(range(len(results['trades'])), results['trades'], width=1, alpha=0.7, color='green')
axes[2].set_ylabel('Trades (shares)')
axes[2].set_xlabel('Time Step')

plt.tight_layout()
plt.show()


## Test 2: Trending Path Robustness

**Hypothesis:** Even when κ is misspecified (MPC assumes mean-reversion but price trends),
the delta exposure should not blow out catastrophically.

**Pass Criteria:** |Δ_portfolio| stays within reasonable bounds (e.g., < 500)


In [ ]:
print("\n" + "="*50)
print("TEST 2: Trending Path Robustness")
print("="*50)

# Generate trending path (strong upward drift, no mean reversion)
drift = 0.5  # 50% annualized drift
prices_trending = generate_trending_path(S0, drift, sigma, dt, n_steps)

# MPC still assumes mean reversion
config_misspec = MPCConfig(kappa=0.2)
mpc_misspec = DeltaHedgeMPC(config_misspec)

# Simulate
results_trend = simulate_gamma_scalping(prices_trending, mpc_misspec, gamma_opt=37.0)

# Log metrics
test_logger.log_metric("Price range min", prices_trending.min())
test_logger.log_metric("Price range max", prices_trending.max())
test_logger.log_metric("Max |Δ_portfolio|", results_trend['max_abs_delta'], "shares")
test_logger.log_metric("Total trades", results_trend['n_trades'])

# Log test result
test2_pass = results_trend['max_abs_delta'] < 500
test_logger.log_case(
    "Trending Path Robustness",
    passed=test2_pass,
    expected="|Δ_portfolio| < 500",
    actual=f"|Δ_portfolio| = {results_trend['max_abs_delta']:.1f}",
    details="Delta should not blow out when κ is misspecified"
)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(prices_trending, label='Price (Trending)', color='steelblue')
axes[0].plot(results_trend['ema'], label='EMA (S̄)', color='orange', linestyle='--')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].set_title('Test 2: Trending Path (κ Misspecified)')

axes[1].plot(results_trend['delta_portfolio'], label='Δ_portfolio', color='red')
axes[1].axhline(y=0, color='black', linestyle=':')
axes[1].axhline(y=500, color='red', linestyle=':', alpha=0.5)
axes[1].axhline(y=-500, color='red', linestyle=':', alpha=0.5)
axes[1].set_ylabel('Delta (shares)')
axes[1].legend()

axes[2].bar(range(len(results_trend['trades'])), results_trend['trades'], width=1, alpha=0.7, color='green')
axes[2].set_ylabel('Trades (shares)')
axes[2].set_xlabel('Time Step')

plt.tight_layout()
plt.show()


## Test 3: Cost Sensitivity

**Hypothesis:** Higher transaction costs should lead to fewer, larger trades.

**Pass Criteria:** Monotonic relationship between cost and trade frequency


In [ ]:
print("\n" + "="*50)
print("TEST 3: Cost Sensitivity")
print("="*50)

# Use the same OU path
np.random.seed(42)
prices_cost_test = generate_ou_path(S0, S_bar, 0.3, sigma, dt, n_steps)

# Test different cost levels
cost_levels = [0.001, 0.01, 0.05, 0.1]
results_by_cost = {}

for c_lin in cost_levels:
    config_cost = MPCConfig(c_lin=c_lin, kappa=0.3)
    mpc_cost = DeltaHedgeMPC(config_cost)
    
    results_by_cost[c_lin] = simulate_gamma_scalping(prices_cost_test, mpc_cost, gamma_opt=37.0)
    
    # Log each cost level result
    test_logger.log_metric(f"Trades @ c_lin={c_lin}", results_by_cost[c_lin]['n_trades'])

# Check monotonicity
trade_counts = [results_by_cost[c]['n_trades'] for c in cost_levels]
is_monotonic = all(trade_counts[i] >= trade_counts[i+1] for i in range(len(trade_counts)-1))

# Log test result
test3_pass = is_monotonic
test_logger.log_case(
    "Cost Sensitivity",
    passed=test3_pass,
    expected="Monotonically decreasing trade count with cost",
    actual=f"Trade counts: {trade_counts}",
    details="Higher costs should lead to fewer trades"
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(cost_levels)), trade_counts, color='steelblue')
axes[0].set_xticks(range(len(cost_levels)))
axes[0].set_xticklabels([str(c) for c in cost_levels])
axes[0].set_xlabel('Linear Cost (c_lin)')
axes[0].set_ylabel('Number of Trades')
axes[0].set_title('Trade Frequency vs Cost')

volumes = [results_by_cost[c]['total_traded'] for c in cost_levels]
axes[1].bar(range(len(cost_levels)), volumes, color='green')
axes[1].set_xticks(range(len(cost_levels)))
axes[1].set_xticklabels([str(c) for c in cost_levels])
axes[1].set_xlabel('Linear Cost (c_lin)')
axes[1].set_ylabel('Total Volume')
axes[1].set_title('Volume vs Cost')

plt.tight_layout()
plt.show()


## Test 4: Constraint Stress Test

**Hypothesis:** When starting near position limits, MPC should:
1. Not exceed q_max
2. Find feasible solutions
3. Gracefully handle the situation

**Pass Criteria:** q stays within bounds, solution is always feasible


In [ ]:
print("\n" + "="*50)
print("TEST 4: Constraint Stress Test")
print("="*50)

# Create scenario where we start near the limit
np.random.seed(42)
prices_stress = generate_ou_path(S0, S_bar, 0.3, sigma, dt, n_steps)

# Start with position near the limit
config_stress = MPCConfig(q_max=200, u_max=50, kappa=0.3)
mpc_stress = DeltaHedgeMPC(config_stress)

# Start near limit with high delta that wants to push us over
results_stress = simulate_gamma_scalping(
    prices_stress, mpc_stress, 
    gamma_opt=50.0,  # Higher gamma = more pressure
    initial_q=150,   # Start near q_max=200
    initial_delta_opt=-300  # Significant delta to hedge
)

# Log metrics
test_logger.log_metric("q_max constraint", config_stress.q_max, "shares")
test_logger.log_metric("Initial q", 150, "shares")
test_logger.log_metric("Max |q| achieved", results_stress['max_q'], "shares")

# Check constraint satisfaction
constraint_satisfied = results_stress['max_q'] <= config_stress.q_max + 1e-6

# Log test result
test4_pass = constraint_satisfied
test_logger.log_case(
    "Constraint Stress",
    passed=test4_pass,
    expected=f"|q| <= {config_stress.q_max}",
    actual=f"|q| = {results_stress['max_q']:.1f}",
    details="Position limits must be respected even under pressure"
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(results_stress['q'], label='q (equity)', color='steelblue')
axes[0].axhline(y=config_stress.q_max, color='red', linestyle='--', label='q_max')
axes[0].axhline(y=-config_stress.q_max, color='red', linestyle='--')
axes[0].set_ylabel('Position (shares)')
axes[0].legend()
axes[0].set_title('Test 4: Constraint Stress (Starting Near Limit)')

axes[1].plot(results_stress['delta_portfolio'], label='Δ_portfolio', color='red')
axes[1].axhline(y=0, color='black', linestyle=':')
axes[1].set_ylabel('Delta (shares)')
axes[1].set_xlabel('Time Step')
axes[1].legend()

plt.tight_layout()
plt.show()


## Summary


In [ ]:
# Generate final summary with all test results
summary = test_logger.log_summary()

# Display summary in a nice format
print("\n" + "="*60)
print("VALIDATION COMPLETE")
print("="*60)
print(f"\nLog files written to: ../logs/")
print("\nTo analyze logs programmatically:")
print("  from src.utils.logging import read_logs")
print("  trades = read_logs('trades')")
print("  mpc_logs = read_logs('mpc')")
print("="*60)
